# Runbook 2.4.c: Motion detectors and sensor-based detection  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks · **Maps to:** 2.4.B, 2.4.C · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** You are building the automated triage logic for a facility's motion-sensor monitoring system at
Crestfield Analytics. Given a list of motion-sensor trigger events, classify each one as a
**TRUE_DETECTION** (a real person likely caused it) or a **FALSE_POSITIVE** (a known nuisance
cause, sunlight, HVAC airflow, a pet, an insect, vibration, or foliage, likely caused it), and
return the IDs of the events that are true detections.

Single-technology sensors (`pir`, `microwave`, `ultrasonic`) report one cause tag. A
**dual-technology** sensor reports two independent cause tags, one from its PIR element, one
from its microwave element, and only counts as a true detection when **both** elements agree the
cause is genuine (AND-logic). This mirrors the lesson: a dual-tech sensor exists specifically to
suppress the false-positive causes that fool a single technology, so if *either* element's cause
is a known false-positive cause, the whole event is a false positive.

**Lesson objective.** Explain how PIR, microwave, and dual-technology motion sensors detect intrusions, evaluate sensor placement and sensitivity tradeoffs, and classify a given sensor trigger event as a true detection or a false positive.

### What comes in
- `events`: `list[dict]`, each dict has:
  - `"id"`: `str`, event identifier, e.g. `"EVT-001"`
  - `"sensor_type"`: `str`, `"pir"`, `"microwave"`, `"ultrasonic"`, or `"dual-tech"`
  - single-tech events also have `"cause"`: `str`
  - dual-tech events instead have `"cause_pir"`: `str` and `"cause_mw"`: `str`
- `false_positive_causes`: `set[str]`, known false-positive cause tags, e.g.
  `{"sunlight", "hvac_airflow", "pet", "insect", "vibration", "foliage"}`

### What must come out
A **sorted list of unique event ID strings** that classify as TRUE_DETECTION. Empty list if none.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
events = [
    {"id": "EVT-001", "sensor_type": "pir", "cause": "person"},
    {"id": "EVT-002", "sensor_type": "pir", "cause": "sunlight"},
]
fp = {"sunlight", "hvac_airflow", "pet", "insect", "vibration", "foliage"}
solve(events, fp)  ->  ["EVT-001"]

events2 = [
    {"id": "EVT-010", "sensor_type": "dual-tech", "cause_pir": "person", "cause_mw": "person"},
    {"id": "EVT-011", "sensor_type": "dual-tech", "cause_pir": "hvac_airflow", "cause_mw": "person"},
]
solve(events2, fp)  ->  ["EVT-010"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Every event has the fields required for its `sensor_type` as described in the Reference section below.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(events: list[dict], false_positive_causes: set[str]) -> list[str]
```

#### Input (`events`)
`list[dict]` where each dict represents one motion-sensor trigger event and has:
- `"id"`: `str`, unique event identifier string
- `"sensor_type"`: `str`, one of `"pir"`, `"microwave"`, `"ultrasonic"`, `"dual-tech"`
- For `sensor_type` in `{"pir", "microwave", "ultrasonic"}` (single-technology sensors):
  - `"cause"`: `str`, the tag describing what triggered the sensor (e.g. `"person"`,
    `"sunlight"`, `"hvac_airflow"`, `"pet"`, `"insect"`, `"vibration"`, `"foliage"`)
- For `sensor_type == "dual-tech"`:
  - `"cause_pir"`: `str`, the cause tag reported by the PIR element
  - `"cause_mw"`: `str`, the cause tag reported by the microwave element

#### Input (`false_positive_causes`)
`set[str]`, the set of cause tags that are known false-positive causes (e.g.
`{"sunlight", "hvac_airflow", "pet", "insect", "vibration", "foliage"}`). Any cause tag NOT in
this set is treated as a genuine, person-consistent cause.

#### Output (`result`)
A **sorted, de-duplicated** `list[str]` of event IDs that classify as **TRUE_DETECTION**. Events
that classify as FALSE_POSITIVE are simply omitted from the result. Empty list if none.

#### Classification rules
- **Single-technology sensor** (`pir`, `microwave`, or `ultrasonic`): the event is a
  TRUE_DETECTION if and only if `"cause"` is NOT in `false_positive_causes`.
- **Dual-technology sensor** (`dual-tech`): AND-logic. The event is a TRUE_DETECTION if and only
  if BOTH `"cause_pir"` and `"cause_mw"` are NOT in `false_positive_causes`. If either element's
  cause is a known false-positive cause, the whole event is a FALSE_POSITIVE, even if the other
  element's cause is genuine.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Every event dict has a well-formed `"id"` and `"sensor_type"` plus the cause field(s) required
  for that sensor type, per the input spec above.
- Event IDs are unique within a single call.
- Output is always sorted ascending and de-duplicated.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import Dict, List, Set


def solve(events: List[Dict], false_positive_causes: Set[str]) -> List[str]:
    true_detections = []

    for e in events:
        sensor_type = e["sensor_type"]

        if sensor_type == "dual-tech":
            pir_ok = e["cause_pir"] not in false_positive_causes
            mw_ok = e["cause_mw"] not in false_positive_causes
            is_true = pir_ok and mw_ok
        else:
            is_true = e["cause"] not in false_positive_causes

        if is_true:
            true_detections.append(e["id"])

    return sorted(set(true_detections))


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

FP = {"sunlight", "hvac_airflow", "pet", "insect", "vibration", "foliage"}


def _single(id_, sensor_type, cause):
    return {"id": id_, "sensor_type": sensor_type, "cause": cause}


def _dual(id_, cause_pir, cause_mw):
    return {"id": id_, "sensor_type": "dual-tech", "cause_pir": cause_pir, "cause_mw": cause_mw}


def test_pir_person_is_true_detection():
    events = [_single("EVT-001", "pir", "person")]
    assert impl.solve(events, FP) == ["EVT-001"]


def test_pir_sunlight_is_false_positive():
    events = [_single("EVT-002", "pir", "sunlight")]
    assert impl.solve(events, FP) == []


def test_microwave_vibration_is_false_positive():
    events = [_single("EVT-003", "microwave", "vibration")]
    assert impl.solve(events, FP) == []


def test_ultrasonic_person_is_true_detection():
    events = [_single("EVT-004", "ultrasonic", "person")]
    assert impl.solve(events, FP) == ["EVT-004"]


def test_dual_tech_both_person_is_true_detection():
    events = [_dual("EVT-005", "person", "person")]
    assert impl.solve(events, FP) == ["EVT-005"]


def test_dual_tech_pir_false_positive_overrides():
    events = [_dual("EVT-006", "hvac_airflow", "person")]
    assert impl.solve(events, FP) == []


def test_dual_tech_mw_false_positive_overrides():
    events = [_dual("EVT-007", "person", "vibration")]
    assert impl.solve(events, FP) == []


def test_empty_events():
    assert impl.solve([], FP) == []


def test_mixed_batch():
    events = [
        _single("EVT-008", "pir", "person"),
        _single("EVT-009", "pir", "pet"),
        _single("EVT-010", "microwave", "hvac_airflow"),
        _dual("EVT-011", "person", "person"),
        _dual("EVT-012", "person", "vibration"),
        _single("EVT-013", "ultrasonic", "insect"),
    ]
    assert impl.solve(events, FP) == ["EVT-008", "EVT-011"]


def test_dual_tech_both_false_positive():
    events = [_dual("EVT-014", "sunlight", "vibration")]
    assert impl.solve(events, FP) == []


def test_single_tech_unknown_cause_treated_as_genuine():
    events = [_single("EVT-015", "pir", "tailgater")]
    assert impl.solve(events, FP) == ["EVT-015"]


def test_output_sorted_and_deduplicated():
    events = [
        _single("EVT-020", "pir", "person"),
        _single("EVT-019", "microwave", "person"),
        _single("EVT-018", "ultrasonic", "person"),
    ]
    result = impl.solve(events, FP)
    assert result == sorted(result)
    assert result == ["EVT-018", "EVT-019", "EVT-020"]


def test_all_sensor_types_mixed_with_custom_fp_set():
    custom_fp = {"sunlight"}
    events = [
        _single("EVT-021", "pir", "hvac_airflow"),
        _single("EVT-022", "pir", "sunlight"),
        _dual("EVT-023", "hvac_airflow", "person"),
    ]
    assert impl.solve(events, custom_fp) == ["EVT-021", "EVT-023"]


def test_no_true_detections_returns_empty_list():
    events = [
        _single("EVT-024", "pir", "pet"),
        _dual("EVT-025", "person", "foliage"),
    ]
    assert impl.solve(events, FP) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it